# Задача 2. NER брендов в пользовательском запросе

Вход — `хочу футболку моспошиф и кеды нью баланс`.

Ожидаемый результат — **два упоминания** с исходным текстом, символьными границами и брендом из справочника.

```python
[
    {"text": "моспошиф", "start": 14, "end": 22, "brand_id": "b_mos"},
    {"text": "нью баланс", "start": 30, "end": 40, "brand_id": "b_nb"},
]
```

Обработаем многословные названия, словоформы, опечатки и разные алфавиты. При близких совпадениях учтём популярность. Используем готовые нормализацию и редакционное сходство из первой задачи.

ID, частоты и запросы учебные. В упрощённой модели один бренд выбирается для каждого непересекающегося фрагмента. Алиас `new` намеренно неоднозначен.

**Версия с решением.** Все ячейки выполнены, результаты сохранены.

In [1]:
import re
import unicodedata
import math
from functools import lru_cache
from rapidfuzz.distance import Levenshtein
from nltk.stem.snowball import SnowballStemmer
from unidecode import unidecode

# ID, название, варианты написания, условная популярность.
brands = [
    ("b_mos", "Моспошив", ["моспошив", "mosposhiv"], 70),
    ("b_nb", "New Balance", ["new balance", "нью баланс", "new"], 650),
    ("b_ne", "New Era", ["new era", "нью эра", "new"], 250),
    ("b_nike", "Nike", ["nike", "найк"], 1100),
    ("b_tnf", "The North Face", ["the north face", "норт фейс"], 600),
    ("b_gold", "Золотое Яблоко", ["золотое яблоко"], 500),
    ("b_samsung", "Samsung", ["samsung", "самсунг"], 1200),
    ("b_samson", "Samson", ["samson", "самсон"], 40),
]

## Шаг 1. Готовая обработка текста

Для каждого слова применяем русский Snowball-стеммер, затем транслитерацию в латиницу. Одинаково преобразуем алиасы и фрагменты запроса. Таблица показывает промежуточные строки. Стемминг может менять имена, поэтому варианты написания храним в справочнике.

Транслитерация и алиасы работают совместно. `нью баланс` связан с `New Balance` через записанный алиас.

In [2]:
stemmer = SnowballStemmer("russian")

def normalize(text):
    text = unicodedata.normalize("NFKC", text).casefold().replace("ё", "е")
    return re.sub(r"[\W_]+", " ", text).strip()

@lru_cache(maxsize=1024)
def search_form(text):
    stems = [stemmer.stem(word) for word in normalize(text).split()]
    return unidecode(" ".join(stems))

for text in ["МОСПОШИВ!", "mosposhiv", "Золотое Яблоко", "Золотого Яблока", "нью баланс"]:
    print(f"{text:20} → {search_form(text)}")
assert search_form("Золотого Яблока") == search_form("Золотое Яблоко")

aliases = [(brand_id, name, search_form(alias), popularity)
           for brand_id, name, variants, popularity in brands for alias in variants]
max_log_popularity = max(math.log1p(popularity) for _, _, _, popularity in brands)

МОСПОШИВ!            → mosposh
mosposhiv            → mosposhiv
Золотое Яблоко       → zolot iablok
Золотого Яблока      → zolot iablok
нью баланс           → n'iu balans


## Шаг 2. Выделяем словные n-граммы · TODO 1

Здесь n-граммы — окна из **1–3 слов**. Размер ограничен самым длинным названием в справочнике. Перебор индексов уже дан.

Заполните три строки внутри цикла — получите `start` первого токена, `end` последнего и добавьте тройку `(start, end, message[start:end])`. Границы считаем по исходному сообщению, правая граница исключительная.

In [3]:
def word_spans(message, max_words=3):
    tokens = list(re.finditer(r"[^\W_]+", message))
    spans = []
    for first in range(len(tokens)):
        for last in range(first, min(first + max_words, len(tokens))):
            start = tokens[first].start()
            end = tokens[last].end()
            spans.append((start, end, message[start:end]))
    return spans

assert (5, 16, "New Balance") in word_spans("хочу New Balance!")
assert word_spans("...") == []
print(word_spans("кеды нью баланс"))

[(0, 4, 'кеды'), (0, 8, 'кеды нью'), (0, 15, 'кеды нью баланс'), (5, 8, 'нью'), (5, 15, 'нью баланс'), (9, 15, 'баланс')]


## Шаг 3. Находим кандидатов · готовый код

Редакционное сходство равно `1 − distance / max_length`. Берём кандидатов со сходством от 0.8. Для коротких строк до трёх символов разрешаем только точное совпадение.

Сравниваем фрагмент с алиасами той же длины в словах. Это уменьшает захват соседних слов. Например, фрагмент `new balance и` не сравнивается с двухсловным `new balance`. Порог 0.8 задан для эксперимента, его можно проверить на своей разметке.

In [4]:
def span_candidates(span, min_similarity=0.8):
    start, end, text = span
    query = search_form(text)
    candidates = []
    for brand_id, name, alias, popularity in aliases:
        if len(query.split()) != len(alias.split()):
            continue
        similarity = Levenshtein.normalized_similarity(query, alias)
        threshold = 1.0 if min(len(query), len(alias)) <= 3 else min_similarity
        if similarity >= threshold:
            candidates.append(dict(start=start, end=end, text=text, brand_id=brand_id,
                                   name=name, similarity=similarity, popularity=popularity))
    return candidates

print(span_candidates((0, 8, "моспошиф")))

[{'start': 0, 'end': 8, 'text': 'моспошиф', 'brand_id': 'b_mos', 'name': 'Моспошив', 'similarity': 0.8888888888888888, 'popularity': 70}]


## Шаг 4. Учитываем популярность · TODO 2

Верните `similarity + weight × log(1 + popularity) / max_log_popularity`. Здесь `math.log1p` считает логарифм, максимальный логарифм уже вычислен.

При `weight=0.03` популярность даёт бонус до 0.03. Сравним кандидатов для короткого `new`. Увеличьте вес и проверьте редкий точный бренд `Samson` рядом с популярным `Samsung`.

In [5]:
def ranking_score(candidate, weight=0.03):
    popularity_score = math.log1p(candidate["popularity"]) / max_log_popularity
    return candidate["similarity"] + weight * popularity_score

assert ranking_score({"similarity": 0.9, "popularity": 0}) == 0.9
assert ranking_score({"similarity": 0.9, "popularity": 1200}, weight=0) == 0.9
new_candidates = sorted(span_candidates((0, 3, "new")), key=ranking_score, reverse=True)
assert new_candidates[0]["brand_id"] == "b_nb"
print([(c["name"], round(ranking_score(c), 3)) for c in new_candidates])

[('New Balance', 1.027), ('New Era', 1.023)]


## Шаг 5. Собираем NER-ответ · TODO 3

Окна перекрываются — в `new balance` найдутся и `new`, и `new balance`. Готовая сортировка сначала рассматривает длинные фрагменты, затем score. Она также объединит повторные совпадения по алиасам при выборе непересекающихся интервалов.

Реализуйте `overlaps` одной строкой. Два полуоткрытых интервала пересекаются, когда начало каждого меньше конца другого. Соприкасающиеся интервалы не пересекаются.

In [6]:
def overlaps(left, right):
    return left["start"] < right["end"] and right["start"] < left["end"]

assert overlaps({"start": 0, "end": 11}, {"start": 0, "end": 3})
assert not overlaps({"start": 0, "end": 3}, {"start": 3, "end": 7})
assert not overlaps({"start": 9, "end": 12}, {"start": 0, "end": 3})

In [7]:
def recognize_brands(message, weight=0.03):
    candidates = [candidate for span in word_spans(message) for candidate in span_candidates(span)]
    ranked = sorted(candidates, key=lambda c: (
        -(c["end"] - c["start"]), -ranking_score(c, weight), c["start"], c["brand_id"]))
    selected = []
    for candidate in ranked:
        if not any(overlaps(candidate, previous) for previous in selected):
            selected.append(candidate)
    return sorted(selected, key=lambda c: c["start"])

message = "хочу футболку моспошиф и кеды нью баланс"
entities = recognize_brands(message)
print([{key: e[key] for key in ["text", "start", "end", "brand_id"]} for e in entities])
assert [e["brand_id"] for e in entities] == ["b_mos", "b_nb"]
assert all(message[e["start"]:e["end"]] == e["text"] for e in entities)
assert recognize_brands("хочу обычную футболку") == []
assert recognize_brands("") == []

[{'text': 'моспошиф', 'start': 14, 'end': 22, 'brand_id': 'b_mos'}, {'text': 'нью баланс', 'start': 30, 'end': 40, 'brand_id': 'b_nb'}]


## Шаг 6. Проверяем точные границы и ID

Считаем Precision и Recall по множествам троек `(start, end, brand_id)`. Совпасть должны и границы, и идентификатор. Строки в разметке ниже нужны только для получения эталонных границ. Функция NER получает целое сообщение.

Это небольшая проверочная выборка. После просмотра её результатов новые изменения проверяйте на дополнительных запросах. Попробуйте повтор одного бренда, сильную опечатку, обычное слово, похожее на бренд.

In [8]:
cases = [
    ("хочу футболку моспошиф", [("моспошиф", "b_mos")]),
    ("кеды нью баланс", [("нью баланс", "b_nb")]),
    ("футболка mosposhiv", [("mosposhiv", "b_mos")]),
    ("из Золотого Яблока", [("Золотого Яблока", "b_gold")]),
    ("куртка the north face", [("the north face", "b_tnf")]),
    ("кеды new", [("new", "b_nb")]),
    ("колонка Samson", [("Samson", "b_samson")]),
    ("nike и моспошив", [("nike", "b_nike"), ("моспошив", "b_mos")]),
    ("нужна обычная футболка", []),
]
true_positive = predicted_count = gold_count = 0
for text, mentions in cases:
    gold = {(text.index(mention), text.index(mention) + len(mention), bid) for mention, bid in mentions}
    predicted = {(e["start"], e["end"], e["brand_id"]) for e in recognize_brands(text)}
    true_positive += len(gold & predicted)
    predicted_count += len(predicted)
    gold_count += len(gold)
    print("OK" if gold == predicted else "РАЗОБРАТЬ", text)
print({"Precision": true_positive / predicted_count if predicted_count else 0,
       "Recall": true_positive / gold_count if gold_count else 0})

OK хочу футболку моспошиф
OK кеды нью баланс
OK футболка mosposhiv
OK из Золотого Яблока
OK куртка the north face
OK кеды new
OK колонка Samson
OK nike и моспошив
OK нужна обычная футболка
{'Precision': 1.0, 'Recall': 1.0}


**Обсудим результат.** Для `new` популярность определяет предпочтение при одинаковом текстовом совпадении. Измените запрос на `кепка new` — наш алгоритм не использует категорию и может выбрать неподходящий бренд. Это отдельное возможное улучшение.

Жадный выбор длинных непересекающихся фрагментов — упрощение. Ошибка в длинном фрагменте может вытеснить корректный короткий. Для прикладного решения нужны размеченные сообщения, включая запросы без брендов.

**Связь с лекцией** — нормализация, n-граммы, стемминг, редакционное расстояние, отбор кандидатов, ранжирование, Precision и Recall.